# StockSense forecasting research: experiments, comparison and uncertainty

This notebook documents the research progression and provides **opt-in, offline** comparison code. It is separate from the authenticated application's saved-XGBoost CSV/JSON inference. Running the default cells reads existing data and reports; it does not download data, fit models, modify prepared datasets or write artifacts.

The objective is a decimal **10-trading-session adjusted-close return**, with an empirical prediction interval. Forecasts are available after the origin session closes. A Transformer was proposed but **was not trained**.

Historical numbers below are recorded experiment results, not outputs recomputed by this notebook. Different snapshots, feature contracts, folds and refit policies must not be combined into a single leaderboard.

## Research progression and evidence

| Stage | Inputs / question | Recorded finding |
|---|---|---|
| Original price-only comparison | Prepared 34-predictor allowlist; zero, Ridge, Random Forest, XGBoost | Validation RMSE: zero 7.1688 pp, Ridge 9.8840, Random Forest 8.1794, XGBoost 7.6646. Learned models initially lost. |
| Early stopping | Stop boosting using development outcomes | One-round XGBoost RMSE 7.1073 pp; training-mean baseline 7.1033 pp. Improvement over zero did not establish feature skill. |
| Relative features | 27 predictors; normalize absolute price indicators | Relative Ridge RMSE 7.1330 pp versus original Ridge 9.8840; still worse than the training-mean baseline. |
| Expanded features | 43 predictors including longer momentum, volatility, money flow and relative strength | Seven-round XGBoost validation RMSE 7.0998 pp; about 0.05% improvement versus mean. Bootstrap intervals included zero. |
| Walk-forward / rolling history | Purged 2023/2024 development evaluations | Small 2024 gains did not repeat reliably in 2023. |
| PCA and compact predictors | Inner-selected PCA components; simplified features | PCA did not consistently beat constants; compact Ridge/XGBoost also lacked consistent improvement. |
| Longer history | Colab Yahoo snapshot, 2010–2024; corrected RSI and volume changes | Seven retrospective years, 2018–2024, provided more market regimes. This longer snapshot is **not** the committed short-history dataset. |
| Ensembles | Fixed 50/50 Ridge–XGBoost and baseline-anchored blend | Small, inconsistent gains; intervals generally included zero. |
| SEC pilot | Acceptance-dated revenue/net income, AAPL/MSFT/NVDA; margin/age features | Matched fundamentals did not consistently help the three-company sample. No universal claim about fundamentals. |
| Additional families | Elastic Net, Extra Trees, Histogram Gradient Boosting | Ridge retained lowest average error in that experiment; all 28 displayed annual 20-session MAE gain intervals crossed zero. |
| Tabular accuracy | Regression predictions converted to UP / NOT UP | Mean/median 56.72% vs XGBoost 56.69%; balanced accuracy about 50%. No directional advantage demonstrated. |
| Repository XGBoost calibration | Explicit XGBoost family; bounded inner grid; refit before separate calibration | Snapshot inference implemented; nominal 90% intervals undercovered in partial 2026. No claim of overall best family or live validated forecasting. |

Earlier experiment notebooks supplied by the user included `StockSense_Forecasting_Experiment.ipynb`, `StockSense_Additional_Models_Experiment (1).ipynb`, and `StockSense_Tabular_Models_Accuracy (1).ipynb`. Their saved outputs/source were inspected when compiling this notebook. Other intermediate results above came from the research conversation; their full original notebooks or data may be unavailable. Exact reproduction requires original source, snapshot, versions, configuration and outputs—not just this narrative.

## Historical additional-model leaderboard

Equal-year average errors for the longer-history 2018–2024 experiment, **historically reported, not recomputed here**:

| Family | MAE (pp) | RMSE (pp) |
|---|---:|---:|
| Ridge | 4.677412 | 6.586109 |
| Histogram Gradient Boosting | 4.683204 | 6.590801 |
| Extra Trees | 4.685253 | 6.597498 |
| Elastic Net | 4.689765 | 6.592175 |
| Training median | 4.692237 | 6.598296 |
| Training mean | 4.693540 | 6.600099 |
| Zero return | 4.746255 | 6.649570 |

Ridge's average MAE gain over median was approximately 0.32%; that alone did not prove a reliable advantage. The original 34 features, the expanded 43-feature snapshot, SEC features and a trained Transformer are **not reconstructed by the illustrative comparison cells below**.

## Setup: discover the checkout and use existing dependencies

Start Jupyter from this checkout, or open the notebook within a checkout. Install the documented offline forecasting requirements in your own environment beforehand if needed. No installation/download cell runs here. `RUN_EXPERIMENTS=False` is the default; the later comparison can take substantial CPU time when enabled.

In [ ]:
from pathlib import Path
import sys, json, hashlib, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/forecasting/inputs.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the StockSense checkout.")
sys.path.insert(0, str(ROOT))
RUN_EXPERIMENTS = False
SAVE_ARTIFACTS = False
AS_OF = "2026-09-18"  # Fixed historical snapshot cutoff; not today's prices.
EVALUATION_YEARS = (2024, 2025, 2026)
from data.forecasting.inputs import build_inputs, RELATIVE_FEATURES
from data.forecasting.sources import load_local
from data.prepare_market_data import PRICE_FEATURES
from data.forecasting.models import _quantile, _scale
print("Execution gates:", RUN_EXPERIMENTS, SAVE_ARTIFACTS)
print({name: importlib.metadata.version(name)
       for name in ("numpy", "pandas", "scikit-learn", "joblib")})

## Load and fingerprint the existing numerical source

Prepared candidates normally live under `data/artifacts/market/price_only_features.parquet`. When absent, use the audited committed `forecast_features.parquet`, whose allowed numerical inputs are equivalent. Raw prices validate prices/targets and provide past-only corrective calculations. Do not silently substitute macro-enriched data.

The historical committed prepared snapshot was 110,229 rows, 52 columns, 87 tickers, dated 2021-09-01–2026-09-18; its raw counterpart had 135,593 rows. Loading a different snapshot can change these values.

In [ ]:
candidate = ROOT / "data/artifacts/market/price_only_features.parquet"
feature_path = candidate if candidate.is_file() else ROOT / "data/processed/forecast_features.parquet"
price_path = ROOT / "data/raw/historical_prices.parquet"
universe_path = ROOT / "data/asset_universe.csv"
features, prices, universe, provenance = load_local(feature_path, price_path, universe_path)
# Relative paths in display; provenance retains hashes for reproducibility.
display(pd.DataFrame([
    {"dataset": name, "file": Path(info["path"]).relative_to(ROOT).as_posix(),
     "rows": info["rows"], "columns": len(info["columns"]), "sha256": info["sha256"]}
    for name, info in provenance["datasets"].items()
]))
frame = build_inputs(features, prices, universe, as_of=AS_OF)
print("Validated relative dataset:", frame.shape)
print("Origin dates:", frame.Date.min(), frame.Date.max())
print("Tickers:", frame.ticker.nunique(), "known targets:", frame.target_return_10d.notna().sum())
display(frame.head())

## Feature and target contracts

`PRICE_FEATURES` is the original **34-column preparation allowlist**, not the experimental relative list. `RELATIVE_FEATURES` is the **27-column forecasting input contract**. Identifiers, target, outcome date and training flag stay outside X.

For ticker-specific ordered prices: `y[t] = adjusted_close[t+10] / adjusted_close[t] - 1`. Backward `return_10d` instead uses `price[t] / price[t-10] - 1`. Relative moving-average features use `price / moving_average - 1`; MACD/ATR are divided by origin price. RSI is corrected to 100 for gains with zero losses and 50 for no movement. Volume change after previous zero volume is undefined and remains missing for training-only imputation.

Same-session SPY/VIX and price indicators require finalized after-close observations. Revisions to adjusted historical prices and the fixed current universe remain provenance limitations.

In [ ]:
print("Original prepared allowlist:", len(PRICE_FEATURES))
print("\n".join(PRICE_FEATURES))
print("\nExperimental relative predictors:", len(RELATIVE_FEATURES))
print("\n".join(RELATIVE_FEATURES))
assert len(RELATIVE_FEATURES) == 27 and len(set(RELATIVE_FEATURES)) == 27
assert not {"Date", "ticker", "target_return_10d", "label_end_date", "is_training_row"}.intersection(RELATIVE_FEATURES)
assert not np.isinf(frame[list(RELATIVE_FEATURES)].to_numpy()).any()
display(frame[list(RELATIVE_FEATURES)].isna().sum().loc[lambda s: s > 0].to_frame("missing"))

## Purged yearly folds

Training precedes July of the previous year; July–September selects parameters; October–December calibrates uncertainty; the following year evaluates. Every earlier row's outcome must mature **strictly before** the next boundary. Current short-history inputs cannot support the 2023 fold with 252 minimum training observations: only 199 purged observations per ticker exist. Consequently the example uses 2024, 2025 and partial 2026.

All these years have already been inspected. They are retrospective development diagnostics, not fresh held-out confirmation. The longer-history 2018–2024 experiments are distinct and require their original Yahoo snapshot.

In [ ]:
TARGET = "target_return_10d"

def make_fold(data, year):
    tuning = pd.Timestamp(year - 1, 7, 1)
    calibration = pd.Timestamp(year - 1, 10, 1)
    evaluation = pd.Timestamp(year, 1, 1)
    end = pd.Timestamp(year + 1, 1, 1)
    known = data[TARGET].notna()
    masks = {
        "training": known & data.Date.lt(tuning) & data.label_end_date.lt(tuning),
        "inner": known & data.Date.ge(tuning) & data.Date.lt(calibration) & data.label_end_date.lt(calibration),
        "calibration": known & data.Date.ge(calibration) & data.Date.lt(evaluation) & data.label_end_date.lt(evaluation),
        "evaluation": known & data.Date.ge(evaluation) & data.Date.lt(end) & data.label_end_date.lt(end),
    }
    counts = data.loc[masks["training"]].groupby("ticker").size()
    eligible = counts[counts >= 252].index
    parts = {name: data.loc[mask & data.ticker.isin(eligible)].sort_values(["Date", "ticker"])
             for name, mask in masks.items()}
    if any(part.empty for part in parts.values()):
        raise ValueError(f"Insufficient rows in fold {year}")
    for name, boundary in (("training", tuning), ("inner", calibration), ("calibration", evaluation), ("evaluation", end)):
        assert parts[name].label_end_date.lt(boundary).all()
    return parts

folds = {year: make_fold(frame, year) for year in EVALUATION_YEARS}
display(pd.DataFrame([
    {"year": year, "partition": name, "rows": len(part), "dates": part.Date.nunique(),
     "first_origin": part.Date.min(), "last_origin": part.Date.max(),
     "latest_outcome": part.label_end_date.max()}
    for year, parts in folds.items() for name, part in parts.items()
]))

## Reusable bounded model definitions — illustrative rerun

These definitions reproduce the **evaluation principles**, not every earlier notebook's precise search. Pipelines fit imputation/scaling only on each training partition. PCA component count is selected on the inner window. A fixed equal-weight blend is defined in advance. The comparison does **not** refit on inner labels; the saved repository walk-forward experiment below does refit before calibration. That distinction can change results.

No Transformer is included. The ensemble includes XGBoost only if its offline package is installed. Missing dependencies are reported explicitly rather than replaced by another estimator.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, balanced_accuracy_score

FEATURES = list(RELATIVE_FEATURES)

def linear(estimator, components=None):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    if components is not None:
        steps.append(("pca", PCA(n_components=components, svd_solver="full")))
    return Pipeline(steps + [("model", estimator)])

def trees(estimator):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])

candidates = {
    "Zero": [DummyRegressor(strategy="constant", constant=0)],
    "Mean": [DummyRegressor(strategy="mean")],
    "Median": [DummyRegressor(strategy="median")],
    "Ridge": [linear(Ridge(alpha=a)) for a in (100., 1000.)],
    "Elastic Net": [linear(ElasticNet(alpha=a, l1_ratio=.5, max_iter=10000)) for a in (.0001, .001, .01)],
    "Random Forest": [trees(RandomForestRegressor(n_estimators=120, max_depth=d, min_samples_leaf=100,
                            max_features=.7, n_jobs=2, random_state=42)) for d in (6, 12)],
    "Extra Trees": [trees(ExtraTreesRegressor(n_estimators=120, max_depth=d, min_samples_leaf=100,
                          max_features=.7, n_jobs=2, random_state=42)) for d in (6, 12)],
    "Histogram GB": [trees(HistGradientBoostingRegressor(max_iter=n, learning_rate=.03, max_leaf_nodes=7,
                          min_samples_leaf=100, l2_regularization=10, early_stopping=False, random_state=42)) for n in (30, 100)],
    "PCA Ridge": [linear(Ridge(alpha=100), components=k) for k in (5, 10, 20)],
}
try:
    from xgboost import XGBRegressor
    candidates["XGBoost"] = [trees(XGBRegressor(n_estimators=n, max_depth=2, learning_rate=.03,
        min_child_weight=100, reg_lambda=20, objective="reg:squarederror", tree_method="hist",
        n_jobs=2, random_state=42)) for n in (1, 10, 50)]
    candidates["50/50 ensemble"] = [VotingRegressor([
        ("ridge", linear(Ridge(alpha=100))), ("xgb", clone(candidates["XGBoost"][1]))], weights=[.5, .5])]
except ImportError:
    print("XGBoost is not installed: its family and blend are unavailable in this example.")
print({family: len(options) for family, options in candidates.items()})
print("Definitions only: nothing fitted.")

## Opt-in comparison and calibration

Set `RUN_EXPERIMENTS=True` deliberately to execute. Each family picks its parameter using inner MAE (RMSE tie-break). Calibration residuals are computed only afterwards; outer outcomes never select a parameter. Every family gets its own uncertainty quantile and uses identical evaluation rows.

`scale = max(annualized_volatility_20 × sqrt(10/252), 0.005)`; `score = abs(error)/scale`; `q` is the finite-sample upper rank at nominal 90%; interval is prediction ± q × scale. This is empirical calibration on dependent, overlapping stock returns: **no IID coverage guarantee**. Width in percentage points is an uncertainty measurement, not confidence or probability of correctness.

In [ ]:
experiment_rows, selected_estimators, predictions = [], {}, {}
if not RUN_EXPERIMENTS:
    print("Skipped fitting. Enable RUN_EXPERIMENTS only for an intentional retrospective rerun.")
else:
    for year, parts in folds.items():
        tr, inner, cal, ev = (parts[name] for name in ("training", "inner", "calibration", "evaluation"))
        majority_up = tr[TARGET].gt(0).mean() > .5
        for family, options in candidates.items():
            scored = []
            for index, option in enumerate(options):
                fitted = clone(option).fit(tr[FEATURES], tr[TARGET])
                p_inner = fitted.predict(inner[FEATURES])
                scored.append((mean_absolute_error(inner[TARGET], p_inner),
                               np.sqrt(mean_squared_error(inner[TARGET], p_inner)), index, fitted))
            inner_mae, inner_rmse, selected_index, fitted = min(scored, key=lambda value: value[:3])
            residuals = np.abs(cal[TARGET].to_numpy() - fitted.predict(cal[FEATURES]))
            q = _quantile(residuals / _scale(cal), .9)
            p = fitted.predict(ev[FEATURES]); y = ev[TARGET].to_numpy()
            half = q * _scale(ev); lower, upper = p-half, p+half
            interval_score = upper-lower + 20*((lower-y)*(y<lower)+(y-upper)*(y>upper))
            experiment_rows.append({"year": year, "model": family, "candidate_index": selected_index,
                "MAE_pp": mean_absolute_error(y,p)*100, "RMSE_pp": np.sqrt(mean_squared_error(y,p))*100,
                "R2": r2_score(y,p), "direction_accuracy_pct": np.mean((y>0)==(p>0))*100,
                "balanced_accuracy_pct": balanced_accuracy_score(y>0,p>0)*100,
                "always_up_accuracy_pct": np.mean(y>0)*100,
                "training_majority_accuracy_pct": np.mean((y>0)==majority_up)*100,
                "interval_coverage": np.mean((y>=lower)&(y<=upper)),
                "width_pp": np.mean(upper-lower)*100, "interval_score_pp": interval_score.mean()*100,
                "inner_MAE_pp": inner_mae*100, "calibration_quantile": q})
            selected_estimators[(year,family)] = fitted
            predictions[(year,family)] = ev[["Date","ticker",TARGET]].assign(prediction=p, lower=lower, upper=upper)
        print(year, "completed")
results = pd.DataFrame(experiment_rows)
if not results.empty:
    display(results.round(4))
    display(results.groupby("model")[["MAE_pp","RMSE_pp","direction_accuracy_pct","balanced_accuracy_pct",
                                      "interval_coverage","width_pp"]].mean().sort_values("MAE_pp").round(4))

## Reading accuracy and selecting a research candidate

Direction is positive return = UP; zero or negative = NOT UP. Accuracy near 56% is not evidence of skill if always-UP scores 56–58%. Balanced accuracy near 50% indicates little discrimination. Return-model selection uses MAE, with RMSE secondary; direction is a separate diagnostic. A baseline can win.

Outer-year tables here are already-inspected development evidence. Choosing a family after examining them requires a new prospective evaluation before any independent superiority claim. Do not retune based on calibration coverage or the final year's score and then call that same year a test.

## Read the actual repository experiment, without unpickling

This cell reads optional `report.json` only. Ignored run artifacts are not expected in a fresh Git clone. If absent, use the static documented historical table below; do not manufacture fresh outputs or silently train. The actual module uses the predeclared 12-configuration XGBoost grid, selects on inner MAE, then refits on labels matured before calibration. XGBoost is explicitly requested; the run does **not** prove it is the overall best family.

In [ ]:
report_path = ROOT / "data/artifacts/forecasting/walkforward-xgboost-2026-09-18/report.json"
repository_report = None
if report_path.is_file():
    repository_report = json.loads(report_path.read_text())
    actual_rows = []
    for fold in repository_report["walk_forward"]:
        metrics = fold["xgboost"]
        actual_rows.append({"year": fold["year"], "selected": fold["metadata"]["selected_model"],
            "MAE_pp": metrics["MAE_pp"], "median_MAE_pp": fold["baselines"]["median"]["MAE_pp"],
            "accuracy_pct": metrics["direction_accuracy_pct"], "always_up_pct": metrics["always_up_accuracy_pct"],
            "coverage": metrics["interval_coverage"], "width_pp": metrics["mean_interval_width_pp"]})
    display(pd.DataFrame(actual_rows).round(4))
    display(pd.DataFrame([{"year": fold["year"], **interval}
        for fold in repository_report["walk_forward"] for interval in fold["paired_mae_intervals"]]))
else:
    print("Ignored repository run report is unavailable. Historical recorded values are documented below.")

### Actual recorded repository results — not recomputed here

| Year | XGBoost MAE (pp) | Median MAE (pp) | Direction / always-UP (%) | Interval coverage | Mean width (pp) |
|---|---:|---:|---:|---:|---:|
| 2024 | 4.685606 | 4.691181 | 57.7230 / 57.7230 | 91.9445% | 23.7303 |
| 2025 | 5.103544 | 5.119377 | 56.6140 / 56.6140 | 92.9262% | 27.4357 |
| 2026, partial | 5.788989 | 5.796326 | 52.6695 / 52.6695 | 86.9278% | 23.9618 |

Selected depth-1 learning-rate .03 models used 10, 10 and 150 boosting rounds respectively. All three had balanced accuracy 50%. MAE gains against median were 0.005575, 0.015833 and 0.007337 pp. The 2024 paired bootstrap intervals excluded zero; 2025/2026 crossed zero for all 10/20/40-session blocks. Repeated experimentation and no multiple-testing correction limit interpretation. Partial 2026 **undercovered** relative to nominal 90%; do not promise calibrated 90% future coverage.

## Dependence-aware diagnostics

Use paired loss differences aggregated by forecast-origin date, then resample contiguous date blocks while keeping all tickers on a date together. Independent row bootstrap would ignore shared market movements and overlapping 10-session outcomes. Monthly/ticker slices can expose failures concealed by averages; origin-time volatility buckets need training-defined thresholds.

The repository's `paired_intervals` implements this approach for its own `ForecastBundle`. No bootstrap runs by default here. The following optional slice cell uses only predictions already generated by the gated comparison.

In [ ]:
if predictions:
    slice_rows = []
    for (year, family), pred in predictions.items():
        for month, group in pred.groupby(pred.Date.dt.to_period("M")):
            error = group[TARGET].to_numpy()-group.prediction.to_numpy()
            slice_rows.append({"year": year, "model": family, "month": str(month), "rows": len(group),
                               "MAE_pp": np.abs(error).mean()*100,
                               "coverage": np.mean((group[TARGET]>=group.lower)&(group[TARGET]<=group.upper))})
    display(pd.DataFrame(slice_rows).head(30).round(4))
else:
    print("No fitted comparison predictions; slice evaluation skipped.")

## Optional exact repository rerun and artifact boundary

The saved repository workflow is separate from the illustrative family comparison above. The next cell is doubly gated, writes only under ignored `data/artifacts/forecasting/`, refuses an existing directory, and performs no downloads. It creates a fresh retrospective report/model/forecasts using the checked-in `walkforward` module. It does not update a production setting or overwrite the application's configured snapshot.

Never load untrusted joblib/pickle artifacts. A model-file hash protects against accidental changes, not executable serialization attacks. Do not commit generated model weights, forecasts, credentials or caches.

In [ ]:
if RUN_EXPERIMENTS and SAVE_ARTIFACTS:
    from datetime import datetime, timezone
    from data.forecasting.walkforward import run
    run_name = "research-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output = ROOT / "data/artifacts/forecasting" / run_name
    run(output, as_of=AS_OF)
else:
    print("No artifacts saved. Both execution gates must be enabled for a repository rerun.")

## Application handoff and remaining work

The authenticated app reads validated **precomputed `forecasts.csv` + `report.json`**. It serves only an explicitly supplied recommender shortlist; unavailable tickers are reported without fabricated forecasts. It does not train, unpickle models, download prices or recalibrate in HTTP requests. Stored snapshot origins are stale and are labeled accordingly.

The separate model bundle holds 27-feature preprocessing, XGBoost, parameter choices, source provenance, cutoff dates and empirical calibration. Refreshing inputs and making fresh inference is a separate operational task; this notebook does not make an old snapshot current. Recalibration after refitting requires new residuals from a model that did not train on those outcomes.

For each row we know finalized past/current price-volume indicators and market context; we try to predict the asset's adjusted-price return over its next 10 sessions. **X = 27 relative forecasting features. y = target_return_10d.** Uncertainty = empirically estimated interval width in percentage points, not accuracy or a probability the point prediction is correct.

Next evidence should be a frozen prospective procedure on genuinely new finalized data, with coverage, width, point errors and always-UP comparisons. No Transformer has been trained, no new dataset is downloaded here, and no independent superiority claim is warranted.

## Manual local deployment package
Production forecasting uses the selected saved XGBoost bundle and an uploaded 27-feature snapshot, not numeric SQL. Local refresh/retraining is explicit through `python -m data.forecasting.local_retrain` with either existing Parquet or `--download-yahoo`. See `data/FORECASTING.md` for completed-session cutoffs, validation/calibration boundaries and package files. No research cell trains in the web request, schedules a download or deploys to Azure. Between uploads, forecasts retain their snapshot origin dates.
